# W07-A · Recommend café items, then real products

Asha, a customer of a café app, loved the iced latte and the cold coffee. The café app has room to suggest one more item. There are two ways to
choose it:

- **content-based**: suggest an item whose **description** is like the ones she loved
- **collaborative filtering**: suggest an item that **customers with the same taste** rated highly

You build both in plain Python on a tiny café menu, then run both on 946 real products and see what each one finds.

By the end of this notebook you will be able to:
1. turn a description into an **item profile**, and recommend the items closest to one a customer loved
2. store a **utility matrix** (the ratings table) as a dictionary, and compare two customers with cosine similarity
3. predict a missing rating with **user–user** filtering
4. flip the table and predict the same rating with **item–item** filtering
5. run both kinds of recommender on real products, and say what each one finds

Cells marked **✍️** are yours to complete. Cells marked **▶** you just run.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shripada/ads6703-nlp/blob/main/w07/W07-A-recommenders.ipynb) · On Colab, run the next cell first. On your laptop, you can skip it.

In [ ]:
# Only for Google Colab: fetch the course folder (with the data) and move into this week's folder.
# On your laptop this cell does nothing.
import os, subprocess, sys
if "google.colab" in sys.modules:
    if os.path.exists("/content/ADS6703"):
        subprocess.run(["git", "-C", "/content/ADS6703", "pull", "-q"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", "https://github.com/shripada/ads6703-nlp.git", "/content/ADS6703"], check=True)
    os.chdir("/content/ADS6703/w07")

In [ ]:
# ▶ Setup: just run this cell. The café's five items, and the four customers' ratings.
import math    # math.sqrt: the square root

# Each item and its description
item_to_description = {
    "iced latte":     "cold coffee with milk",
    "cold coffee":    "cold black coffee",
    "hot latte":      "hot coffee with milk",
    "masala chai":    "hot spiced tea with milk",
    "chocolate cake": "sweet chocolate cake",
}

# The telling words: an item's profile has one number for each of them
feature_words = ["coffee", "tea", "milk", "cold", "hot", "sweet"]

# The ratings table. Each customer has a dictionary: item -> rating, from 1 to 5.
# An item that is missing from a customer's dictionary is a blank: not rated yet.
customer_to_ratings = {
    "Asha":  {"iced latte": 5, "cold coffee": 5, "masala chai": 2},
    "Ravi":  {"iced latte": 5, "cold coffee": 4, "hot latte": 3, "masala chai": 1, "chocolate cake": 5},
    "Meera": {"iced latte": 4, "cold coffee": 4, "hot latte": 4, "masala chai": 3, "chocolate cake": 3},
    "Tom":   {"iced latte": 1, "cold coffee": 2, "hot latte": 4, "masala chai": 5, "chocolate cake": 3},
}
print(len(item_to_description), "items,", len(customer_to_ratings), "customers")

## 1 · Item profiles

An item's **profile** is its description written as numbers: for each feature word, **1** if the word is in the
description, **0** if not.

| | coffee | tea | milk | cold | hot | sweet |
|---|---|---|---|---|---|---|
| iced latte: *cold coffee with milk* | 1 | 0 | 1 | 1 | 0 | 0 |

### ✍️ Exercise 1: `item_profile(description)`

Split the description into words. Then, for each feature word, add 1 to the profile if it is one of those words, and
0 if not.

In [ ]:
def item_profile(description):
    description_words = description.split()   # a list of strings, e.g. ["cold", "coffee", "with", "milk"]
    profile = []                               # a list of numbers: one per feature word
    for feature_word in feature_words:
        if feature_word in description_words:
            ...   # ✍️ add 1 to the profile
        else:
            ...   # ✍️ add 0 to the profile
    return profile
# Check: the profile of the iced latte
print(item_profile("cold coffee with milk"))
assert item_profile("cold coffee with milk") != [], "item_profile returned an empty list: fill in the two ✍️ lines"
assert item_profile("cold coffee with milk") == [1, 0, 1, 1, 0, 0], "not quite: compare with the table above"
assert item_profile("a glass of water") == [0, 0, 0, 0, 0, 0], "no feature word: all 0"
print("✅")

In [ ]:
# ▶ Just run this cell: the profile of every item, in one dictionary (item -> its profile).
item_to_profile = {}
for item, description in item_to_description.items():
    item_to_profile[item] = item_profile(description)

print(f"{'':15s}", feature_words)
for item, profile in item_to_profile.items():
    print(f"{item:15s}", profile)

To compare two profiles we use **cosine similarity**: the dot product divided by both lengths. 1 means the same words,
0 means no word in common. Just run the cell.

In [ ]:
# ▶ Just run this cell: cosine similarity of two lists of numbers.
def dot(list_a, list_b):
    # multiply the numbers in the same position, then add them all up
    total = 0
    for number_a, number_b in zip(list_a, list_b):   # zip pairs them up: (list_a[0], list_b[0]), ...
        total = total + number_a * number_b
    return total

def length(numbers):
    return math.sqrt(dot(numbers, numbers))          # √(sum of squares)

def cosine(list_a, list_b):
    if length(list_a) == 0 or length(list_b) == 0:
        return 0                                     # all 0s: nothing in common
    return dot(list_a, list_b) / (length(list_a) * length(list_b))

print("iced latte and hot latte:", round(cosine(item_to_profile["iced latte"], item_to_profile["hot latte"]), 2))

> 📖 **Reading the output.** 0.67: the two lattes share *coffee* and *milk*, but one is *cold* and the other *hot*.

## 2 · Recommend from descriptions

A content-based recommender takes an item the customer loved, and ranks the items she has **not** rated yet by their
cosine with it.

### ✍️ Exercise 2: `recommend_by_description(loved_item, customer)`

Go through every item. Skip the items the customer has already rated. For each of the others, add the pair
(cosine with the loved item, item) to the list. Then sort the list, biggest cosine first.

In [ ]:
def recommend_by_description(loved_item, customer):
    rated_items = customer_to_ratings[customer]           # a dictionary: item -> rating
    cosine_and_item = []                                  # a list of (cosine, item) tuples
    for item, profile in item_to_profile.items():
        if item in rated_items:
            continue                                      # already rated: skip it
        similarity = ...   # ✍️ the cosine of the loved item's profile and this profile
        cosine_and_item.append((round(similarity, 2), item))
    return sorted(cosine_and_item, reverse=True)          # biggest cosine first
# Check: Asha loved the iced latte. What do her descriptions suggest?
try:
    suggestions_for_asha = recommend_by_description("iced latte", "Asha")
except TypeError:
    raise AssertionError("fill in the ✍️ line") from None
print(suggestions_for_asha)
assert suggestions_for_asha == [(0.67, "hot latte"), (0.0, "chocolate cake")], "not quite"
print("✅")

> 📖 **Reading the output.** Asha has not had the hot latte or the chocolate cake. The descriptions pick the **hot
> latte**: it shares *coffee* and *milk* with the iced latte. The cake shares no word, so it gets 0, and a
> content-based recommender will never suggest it to her.

### A new item

The café adds an **iced mocha**, *cold sweet coffee with milk*. Nobody has rated it yet, but its description is ready
on day one. Just run the cell.

In [ ]:
# ▶ Just run this cell: add the iced mocha, and recommend for Asha again.
item_to_description["iced mocha"] = "cold sweet coffee with milk"
item_to_profile["iced mocha"] = item_profile(item_to_description["iced mocha"])
print("iced mocha profile:", item_to_profile["iced mocha"])
print(recommend_by_description("iced latte", "Asha"))

> 📖 **Reading the output.** The iced mocha goes straight to the top, at 0.87, though no customer has rated it. Keep it
> in mind: in section 4, the ratings cannot do this.

## 3 · Collaborative filtering: the ratings table

The four customers' ratings form a table: one row per customer, one column per item. This table is called the
**utility matrix**. A dot **·** is a blank. Just run the cell.

In [ ]:
# ▶ Just run this cell: print the utility matrix.
items = list(item_to_description)          # every item, in the same order as the dictionary
print(f"{'':7s}" + "".join(f"{item:>16s}" for item in items))
for customer, ratings in customer_to_ratings.items():
    cells = [str(ratings[item]) if item in ratings else "·" for item in items]   # a rating, or · for a blank
    print(f"{customer:7s}" + "".join(f"{cell:>16s}" for cell in cells))

> 📖 **Reading the output.** Asha has two blanks among the old items (the hot latte and the cake), and nobody has rated
> the iced mocha. Collaborative filtering fills the blanks, then suggests the highest.

To compare two customers, compare their rows with cosine similarity. A blank counts as **0**:

- the **dot product** only adds up the items **both** have rated (a blank times anything is 0)
- each **length** uses **every** rating in that customer's row

### ✍️ Exercise 3: `similarity(ratings_a, ratings_b)`

Each argument is a dictionary, key → rating. Add up rating × rating for the keys in both dictionaries. Then divide by
the two lengths.

In [ ]:
def similarity(ratings_a, ratings_b):
    # ratings_a and ratings_b are dictionaries: key -> rating. A missing key is a blank, i.e. 0.
    dot_product = 0
    for key, rating_a in ratings_a.items():
        if key in ratings_b:                                       # only keys both have count
            dot_product = ...   # ✍️ add rating_a times the other rating for this key
    length_a = math.sqrt(sum(rating * rating for rating in ratings_a.values()))   # every rating in the row
    length_b = math.sqrt(sum(rating * rating for rating in ratings_b.values()))
    if length_a == 0 or length_b == 0:
        return 0
    return dot_product / (length_a * length_b)
# Check: Asha against Meera
try:
    asha_meera = similarity(customer_to_ratings["Asha"], customer_to_ratings["Meera"])
except TypeError:
    raise AssertionError("fill in the ✍️ line") from None
print("Asha and Meera:", round(asha_meera, 2))
assert round(asha_meera, 2) == 0.77, "not quite: the dot product is 46, the lengths √54 and √66"
print("✅")

In [ ]:
# ▶ Just run this cell: Asha against every other customer.
for customer, ratings in customer_to_ratings.items():
    if customer != "Asha":
        print(f"Asha and {customer:6s}", round(similarity(customer_to_ratings["Asha"], ratings), 2))

> 📖 **Reading the output.** Meera (0.77) and Ravi (0.73) rate most like Asha. Tom (0.46) loves the chai that Asha
> rated 2, yet he still gets 0.46: every rating is positive, so every cosine is positive. Section 5 fixes that.

## 4 · User–user: fill a blank from the customers most like you

To predict Asha's rating for an item:

1. take every other customer who has rated that item, with their similarity to Asha
2. keep the **two** most similar: the ones with the highest cosine similarity
3. take the **weighted average** of their ratings: closer customers count for more

$$\text{prediction} = \frac{\text{sim}_1 \times \text{rating}_1 + \text{sim}_2 \times \text{rating}_2}{\text{sim}_1 + \text{sim}_2}$$

### ✍️ Exercise 4: `predict_user_user(customer, item)`

Steps 1 and 2 are written. Fill in the two sums of the weighted average: `similarity_into_rating_sum` (the top) and `total_similarity` (the bottom).

In [ ]:
def predict_user_user(customer, item, top=2):
    similarity_and_rating = []                       # a list of (similarity, their rating of the item) tuples
    for other_customer, other_ratings in customer_to_ratings.items():
        if other_customer != customer and item in other_ratings:
            sim = similarity(customer_to_ratings[customer], other_ratings)
            similarity_and_rating.append((sim, other_ratings[item]))
    closest = sorted(similarity_and_rating, reverse=True)[:top]   # the `top` most similar customers
    similarity_into_rating_sum = 0
    total_similarity = 0
    for sim, rating in closest:
        similarity_into_rating_sum = ...   # ✍️ add similarity times rating
        total_similarity = ...             # ✍️ add the similarity
    return similarity_into_rating_sum / total_similarity if total_similarity != 0 else 0   # 0 if no one rated it
# Check: Asha and the chocolate cake
try:
    cake_for_asha = predict_user_user("Asha", "chocolate cake")
except TypeError:
    raise AssertionError("fill in the two ✍️ lines") from None
print("Asha's chocolate cake:", round(cake_for_asha, 2))
print("Asha's hot latte:     ", round(predict_user_user("Asha", "hot latte"), 2))
assert abs(cake_for_asha - 3.97) < 0.02, "not quite: Meera (0.77) gave it 3, Ravi (0.73) gave it 5"
print("✅")

> 📖 **Reading the output.** The ratings predict about **4.0** for the cake (3.98; by hand, with cosines rounded to
> 0.77 and 0.73, it comes to 3.97) and **3.5** for the hot latte, so they pick
> the **cake**. The descriptions picked the hot latte. No description says that coffee lovers like cake; the ratings
> of Ravi and Meera do.

Now try the iced mocha. Just run the cell.

In [ ]:
# ▶ Just run this cell: nobody has rated the iced mocha.
try:
    print(predict_user_user("Asha", "iced mocha"))
except ZeroDivisionError:
    print("No customer has rated the iced mocha: nothing to average. This is the cold start problem.")

## 5 · Item–item: fill a blank from items rated alike

Now read the same table **by columns**. Each item becomes a dictionary, customer → rating. To predict Asha's rating
for the cake:

1. compare the cake's column with the column of every item **Asha has rated**
2. keep the two most similar items
3. take the weighted average of **Asha's own** ratings of those items

First, flip the table.

### ✍️ Exercise 5: `to_item_to_ratings(customer_to_ratings)`

Build a new dictionary, item → {customer: rating}. For every customer and every item they rated, put the rating into
that item's dictionary.

In [ ]:
def to_item_to_ratings(customer_to_ratings):
    item_to_ratings = {}                                 # a dictionary: item -> {customer: rating}
    for customer, ratings in customer_to_ratings.items():
        for item, rating in ratings.items():
            if item not in item_to_ratings:
                item_to_ratings[item] = {}               # a new item: an empty dictionary
            ...   # ✍️ put the rating into this item's dictionary, under this customer
    return item_to_ratings
item_to_ratings = to_item_to_ratings(customer_to_ratings)
# Check: the cake's column
print("chocolate cake:", item_to_ratings["chocolate cake"])
assert item_to_ratings["chocolate cake"] != {}, "the cake's dictionary is empty: fill in the ✍️ line"
assert item_to_ratings["chocolate cake"] == {"Ravi": 5, "Meera": 3, "Tom": 3}, "not quite"
print("✅")

Your `similarity` from Exercise 3 works on columns too: it only needs two dictionaries. Just run the cell.

In [ ]:
# ▶ Just run this cell: the item–item prediction, the mirror image of predict_user_user.
def predict_item_item(customer, item, top=2):
    similarity_and_rating = []                       # a list of (similarity, the customer's own rating) tuples
    for rated_item, rating in customer_to_ratings[customer].items():   # only items this customer has rated
        sim = similarity(item_to_ratings[item], item_to_ratings[rated_item])
        similarity_and_rating.append((sim, rating))
    closest = sorted(similarity_and_rating, reverse=True)[:top]
    similarity_into_rating_sum = sum(sim * rating for sim, rating in closest)
    total_similarity = sum(sim for sim, rating in closest)
    return similarity_into_rating_sum / total_similarity

for rated_item in customer_to_ratings["Asha"]:
    print(f"cake and {rated_item:12s}", round(similarity(item_to_ratings["chocolate cake"], item_to_ratings[rated_item]), 2))
print()
print("Asha's chocolate cake, item–item:", round(predict_item_item("Asha", "chocolate cake"), 2))
print("Asha's chocolate cake, user–user:", round(predict_user_user("Asha", "chocolate cake"), 2))

> 📖 **Reading the output.** The cake's column is closest to the iced latte (0.75) and the cold coffee (0.74), which
> Asha rated 5 and 5, so item–item predicts **5.0**. User–user said about 4.0. Different numbers, the same decision:
> **recommend the cake**.

### Harsh raters: subtract each customer's average (centring)

Tom's tastes are opposite to Asha's, yet their cosine was 0.46. Subtract each customer's average rating from each of
their ratings: now + means *liked more than usual*, − means *less than usual*, and a blank means *no opinion*. Just
run the cell.

In [ ]:
# ▶ Just run this cell: centre every row, then compare Asha and Tom again.
def centre(ratings):
    # ratings is a dictionary: item -> rating. Returns the same items, each minus the customer's average.
    average = sum(ratings.values()) / len(ratings)
    return {item: rating - average for item, rating in ratings.items()}

print("Asha centred:", centre(customer_to_ratings["Asha"]))
print("Tom centred: ", centre(customer_to_ratings["Tom"]))
print()
for customer in ["Meera", "Ravi", "Tom"]:
    raw = similarity(customer_to_ratings["Asha"], customer_to_ratings[customer])
    centred = similarity(centre(customer_to_ratings["Asha"]), centre(customer_to_ratings[customer]))
    print(f"Asha and {customer:6s} raw {raw:5.2f}   centred {centred:5.2f}")

> 📖 **Reading the output.** After centring, Tom drops from 0.46 to **−0.90**: opposite tastes, as they should be.
> Ravi moves closer to Asha (0.85), and Meera stays close (0.75). Centring changes most where raw ratings mislead.

### Try your own

Add yourself as a customer: rate two or three of the items, from 1 to 5, and run the cell. Which item does each
method suggest for you?

In [ ]:
my_ratings = {"iced latte": 2, "masala chai": 5}    # ✍️ try your own: item -> rating, from 1 to 5

customer_to_ratings["Me"] = my_ratings
item_to_ratings = to_item_to_ratings(customer_to_ratings)   # flip again, so the columns include your ratings
for item in ["iced latte", "cold coffee", "hot latte", "masala chai", "chocolate cake"]:
    if item not in my_ratings:
        print(f"{item:15s} user–user {predict_user_user('Me', item):.2f}   item–item {predict_item_item('Me', item):.2f}")
del customer_to_ratings["Me"]                         # take yourself out again
item_to_ratings = to_item_to_ratings(customer_to_ratings)

## 6 · Both kinds on 946 real products

Now real data: **30,000 Amazon reviews** of home appliances and parts. Each review has a customer (`user_id`), a
product (`parent_asin`, a product code) and a rating from 1 to 5. Each product has a title and a description. Just run
the cell.

In [ ]:
# ▶ Just run this cell: load the reviews and products, and build the real ratings table.
import csv, gzip

DATA = "../data/"   # notebooks live in ADS6703/w07/, data in ADS6703/data/
with gzip.open(DATA + "reviews.csv.gz", "rt", encoding="utf-8") as file:
    reviews = list(csv.DictReader(file))          # a list of dictionaries, one per review
with gzip.open(DATA + "products.csv.gz", "rt", encoding="utf-8") as file:
    products = list(csv.DictReader(file))         # a list of dictionaries, one per product

code_to_title = {product["parent_asin"]: product["title"][:70] for product in products}   # product code -> its name

# The same shape as the café table: customer -> {product code: rating}
real_customer_to_ratings = {}
for review in reviews:
    customer, product_code = review["user_id"], review["parent_asin"]
    if customer not in real_customer_to_ratings:
        real_customer_to_ratings[customer] = {}
    real_customer_to_ratings[customer][product_code] = int(review["rating"])

product_to_ratings = to_item_to_ratings(real_customer_to_ratings)   # your function from Exercise 5: product -> {customer: rating}

cells = len(real_customer_to_ratings) * len(product_to_ratings)
print(len(real_customer_to_ratings), "customers ×", len(product_to_ratings), "products =", cells, "cells")
print(len(reviews), "of them filled:", round(100 * len(reviews) / cells, 2), "%")

> 📖 **Reading the output.** More than ten million cells, and fewer than 3 in 1,000 are filled. Most customers rated
> only one or three of the 946 products. Two products can only be compared through the few customers who rated
> **both**.

### From the descriptions

Each product's title, features and description become a profile. With thousands of different words, real profiles use
**TF-IDF** weights instead of 1 and 0: a word that is rare across the products, like *frothing*, counts for more than
one that is everywhere, like *quality*. The scikit-learn library builds them in one line. Just run the cell.

In [ ]:
# ▶ Just run this cell: TF-IDF profiles of all 946 products, and their cosine similarities.
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

product_codes = [product["parent_asin"] for product in products]
product_texts = [product["title"] + " " + product["features"] + " " + product["description"] for product in products]

# stop_words="english" drops words like "the" and "with"; min_df=2 drops words found in only one product
profiles = TfidfVectorizer(stop_words="english", min_df=2).fit_transform(product_texts)
description_cosines = cosine_similarity(profiles)   # a 946 × 946 table: the cosine of every pair of products
print("profiles:", profiles.shape[0], "products ×", profiles.shape[1], "words")

def similar_by_description(product_code, top=3):
    row = description_cosines[product_codes.index(product_code)]       # this product's cosines with every product
    ranked = sorted(zip(row, product_codes), reverse=True)              # (cosine, code) pairs, biggest first
    return [(round(float(cos), 2), code_to_title[code]) for cos, code in ranked if code != product_code][:top]

### From the ratings

Item–item on the real table: your `similarity` from Exercise 3, on product columns. It also counts how many customers
rated both products. Just run the cell.

In [ ]:
# ▶ Just run this cell: the products whose rating columns are most like a given product's.
def similar_by_ratings(product_code, top=3):
    column = product_to_ratings[product_code]                 # a dictionary: customer -> rating
    results = []                                              # a list of (similarity, shared customers, title)
    for other_code, other_column in product_to_ratings.items():
        shared = len(set(column) & set(other_column))         # customers who rated both
        if other_code != product_code and shared > 0:
            results.append((round(similarity(column, other_column), 2), shared, code_to_title[other_code]))
    return sorted(results, reverse=True)[:top]

def compare(product_code):
    print("##", code_to_title[product_code], f"({len(product_to_ratings[product_code])} ratings)")
    print("From the descriptions:")
    for cos, title in similar_by_description(product_code):
        print(f"   {cos:.2f}  {title}")
    print("From the ratings:")
    for sim, shared, title in similar_by_ratings(product_code):
        print(f"   {sim:.2f}  {shared:2d} shared customers  {title}")
    print()

compare("B07QV3TH8K")   # a milk frothing pitcher
compare("B01H4A4KY0")   # foil seals for refilling Nespresso Vertuo capsules

> 📖 **Reading the output.** For the **frothing pitcher**, the descriptions find other frothing pitchers. The ratings,
> with only two or three customers in common, find a fridge water filter and a coffee filter: **noise**, from a table
> that is almost empty.
>
> For the **foil seals**, the descriptions find foil seals of other brands: **look-alikes**, products you could buy
> *instead*. The ratings find the same brand's capsule kit, paper filters and silicone caps: **companions**, products
> bought *together*. No description would link a foil seal to a silicone cap.
>
> That is why a shop page has two boxes: *"Similar items"* from descriptions, *"Frequently bought together"* from
> customers.

### Try your own

Search the product titles for a word, pick a code from the list, and compare. Try *aeropress*, *mwf*, *keurig* or
*ice maker*.

In [ ]:
search_word = "aeropress"    # ✍️ try your own

for code, title in code_to_title.items():
    if search_word.lower() in title.lower():
        print(code, f"({len(product_to_ratings.get(code, {}))} ratings)", title)

In [ ]:
compare("B0BMV7X9YJ")    # ✍️ paste a code from the list above

## Recap

| idea | what you saw |
|---|---|
| Item profile | a description as numbers: 1 if a feature word is there, 0 if not |
| Content-based | the closest item not yet tried: the **hot latte** for Asha; works for the new iced mocha |
| Utility matrix | a row per customer, a column per item; a blank counts as 0 |
| User–user | the two customers most like Asha, a weighted average of their ratings: cake ≈ 4.0 |
| Item–item | flip the table, the two items most like the cake, Asha's own ratings: cake = 5.0 |
| Centring | subtract each customer's average: Tom goes from 0.46 to −0.90 |
| Cold start | no ratings, no prediction: the iced mocha |
| Real products | descriptions find look-alikes; ratings find companions, or noise when few customers are shared |